# Prototype: Transformer Extraction for Project 042-2024/25

This notebook extracts five prototype data points from the four files for project `042-2024/25`:

1. project identity
2. project runtime and location
3. target group
4. main Soll/Ist performance metric
5. project measures / summary

The prototype uses deterministic document parsing plus transformer-based German label matching and GLiNER-based entity extraction from German Word text.

## 0. Optional dependency install

Run this cell if `sentence-transformers` or `gliner` are not installed in your virtual environment.

```bash
%pip install sentence-transformers gliner torch sqlalchemy pydantic rapidfuzz
```

In [3]:
from pathlib import Path
import json
import math
import re
from datetime import datetime

import pandas as pd
from openpyxl import load_workbook
from docx import Document

try:
    from sentence_transformers import SentenceTransformer, util
except ImportError as exc:
    raise ImportError("Install sentence-transformers first: %pip install sentence-transformers") from exc

try:
    from gliner import GLiNER
except ImportError as exc:
    raise ImportError("Install GLiNER first: %pip install gliner") from exc

[ERROR] `cache_position` is part of T5Model.forward's signature, but not documented. Make sure to add it to the docstring of the function in /Users/tr/Coding/Studies/PublicAIHackathonTeam3/.venv/lib/python3.13/site-packages/transformers/models/t5/modeling_t5.py.
[ERROR] `cache_position` is part of T5ForConditionalGeneration.forward's signature, but not documented. Make sure to add it to the docstring of the function in /Users/tr/Coding/Studies/PublicAIHackathonTeam3/.venv/lib/python3.13/site-packages/transformers/models/t5/modeling_t5.py.


In [8]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "Hackathon" / "2024_2025"

PROJECT_DESCRIPTION_DOCX = DATA_DIR / "042_HandWerk.Zukunft_2024_2025_Projektbeschreibung.docx"
FINAL_REPORT_DOCX = DATA_DIR / "042_HandWerk.Zukunft-2024_2025_Inhaltlicher Endbericht.docx"
SOLL_XLSX = DATA_DIR / "042_HandWerk.Zukunft_2024_25_Indikatorenblatt.xlsx"
IST_XLSX = DATA_DIR / "042_HandWerk.Zukunft-2024-2025_Indikatorenbericht.xlsx"

for path in [PROJECT_DESCRIPTION_DOCX, FINAL_REPORT_DOCX, SOLL_XLSX, IST_XLSX]:
    assert path.exists(), f"Missing file: {path}"

print("Using files:")
for path in [PROJECT_DESCRIPTION_DOCX, FINAL_REPORT_DOCX, SOLL_XLSX, IST_XLSX]:
    print("-", path.relative_to(ROOT))

Using files:
- Hackathon/2024_2025/042_HandWerk.Zukunft_2024_2025_Projektbeschreibung.docx
- Hackathon/2024_2025/042_HandWerk.Zukunft-2024_2025_Inhaltlicher Endbericht.docx
- Hackathon/2024_2025/042_HandWerk.Zukunft_2024_25_Indikatorenblatt.xlsx
- Hackathon/2024_2025/042_HandWerk.Zukunft-2024-2025_Indikatorenbericht.xlsx


## 1. Load transformer models

`paraphrase-multilingual-MiniLM-L12-v2` is a compact multilingual embedding model and works well for German label similarity. GLiNER is used with German entity labels for the Word documents.

In [10]:
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
GLINER_MODEL_NAME = "urchade/gliner_multi-v2.1"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
gliner_model = GLiNER.from_pretrained(GLINER_MODEL_NAME)

print("Loaded:", EMBEDDING_MODEL_NAME)
print("Loaded:", GLINER_MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4784.23it/s]
/Users/tr/Coding/Studies/PublicAIHackathonTeam3/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 37854.73it/s]
[transformers] Could not extract SentencePiece model from /Users/tr/.cache/huggingface/hub/models--microsoft--mdeberta-v3-base/snapshots/a0484667b22365f84929a935b5e50a51f71f159d/spm.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to Ti

ValueError: Error parsing line b'\x0e' in /Users/tr/.cache/huggingface/hub/models--microsoft--mdeberta-v3-base/snapshots/a0484667b22365f84929a935b5e50a51f71f159d/spm.model

## 2. Helper functions

In [5]:
def clean_text(value):
    if value is None:
        return ""
    if isinstance(value, datetime):
        return value.date().isoformat()
    text = str(value).replace("\n", " ").strip()
    return re.sub(r"\s+", " ", text)


def parse_excel_date(value):
    if isinstance(value, datetime):
        return value.date().isoformat()
    text = clean_text(value)
    if not text:
        return None
    if "00:00:00" in text:
        return text.split()[0]
    return text


def cosine_best(query, candidates, min_score=0.45):
    if not candidates:
        return None
    query_embedding = embedding_model.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    candidate_embeddings = embedding_model.encode(candidates, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(query_embedding, candidate_embeddings)[0]
    best_idx = int(scores.argmax())
    best_score = float(scores[best_idx])
    if best_score < min_score:
        return None
    return best_idx, best_score


def non_empty_cells(ws, max_row=None, max_col=None):
    max_row = max_row or ws.max_row
    max_col = max_col or ws.max_column
    cells = []
    for row in ws.iter_rows(min_row=1, max_row=max_row, min_col=1, max_col=max_col):
        for cell in row:
            text = clean_text(cell.value)
            if text:
                cells.append({
                    "sheet": ws.title,
                    "row": cell.row,
                    "col": cell.column,
                    "coordinate": cell.coordinate,
                    "text": text,
                    "value": cell.value,
                })
    return cells


def nearest_right_value(ws, row, col, max_steps=8):
    for next_col in range(col + 1, min(ws.max_column, col + max_steps) + 1):
        value = ws.cell(row=row, column=next_col).value
        if clean_text(value):
            return value, ws.cell(row=row, column=next_col).coordinate
    return None, None


def extract_key_value_semantic(workbook_path, sheet_name, german_query, max_row=40, max_col=20, min_score=0.50):
    wb = load_workbook(workbook_path, data_only=True, read_only=False)
    ws = wb[sheet_name]
    cells = non_empty_cells(ws, max_row=max_row, max_col=max_col)
    texts = [cell["text"] for cell in cells]
    best = cosine_best(german_query, texts, min_score=min_score)
    if best is None:
        return {"value": None, "confidence": 0.0, "evidence": None}
    idx, score = best
    label_cell = cells[idx]
    value, value_coordinate = nearest_right_value(ws, label_cell["row"], label_cell["col"])
    return {
        "value": clean_text(value),
        "confidence": round(score, 3),
        "evidence": {
            "source_file": str(workbook_path.relative_to(ROOT)),
            "sheet": sheet_name,
            "matched_label": label_cell["text"],
            "label_cell": label_cell["coordinate"],
            "value_cell": value_coordinate,
        },
    }

## 3. Extract project identity, runtime, and location from Excel

All matching queries are German. The code uses embeddings to match our canonical German field names to the actual labels in the spreadsheet.

In [6]:
overview_sheet = "Overview IB"

excel_fields = {
    "projekttraeger": "Projektträger oder Trägerorganisation",
    "projekttitel": "Projekttitel oder Projektname",
    "projektnummer": "Projektnummer oder Geschäftszeichen",
    "laufzeit_beginn": "Laufzeit Beginn oder Projektstart",
    "laufzeit_ende": "Laufzeit Ende oder Projektende",
    "hauptprojektstandort": "Hauptprojektstandort oder Bundesland des Projekts",
}

project_meta = {}
for field_name, german_query in excel_fields.items():
    project_meta[field_name] = extract_key_value_semantic(
        IST_XLSX,
        overview_sheet,
        german_query,
        max_row=25,
        max_col=12,
        min_score=0.48,
    )

project_meta["laufzeit_beginn"]["value"] = parse_excel_date(project_meta["laufzeit_beginn"]["value"])
project_meta["laufzeit_ende"]["value"] = parse_excel_date(project_meta["laufzeit_ende"]["value"])

pd.DataFrame([
    {
        "field": key,
        "value": item["value"],
        "confidence": item["confidence"],
        "matched_label": item["evidence"]["matched_label"] if item["evidence"] else None,
        "cell": item["evidence"]["value_cell"] if item["evidence"] else None,
    }
    for key, item in project_meta.items()
])

/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,field,value,confidence,matched_label,cell
0,projekttraeger,Steiermark,0.710,Standort Projektmanagement,D7
1,projekttitel,HandWerk.Zukunft – Berufsvorbereitungskurse mi...,0.697,Projekttitel,D5
2,projektnummer,042-2024/25,0.844,Projektnummer,D6
3,laufzeit_beginn,2024-01-01,0.818,Laufzeit Beginn,D9
4,laufzeit_ende,2025-12-31,0.794,Laufzeit Ende,D10
5,hauptprojektstandort,Steiermark,0.901,Hauptprojektstandort,D8


## 4. Extract the main Soll/Ist KPI from the final indicator report

Prototype KPI: `Anzahl der Projektteilnehmenden gesamt`.

This combines:

- semantic row matching for the indicator name
- semantic column/header matching for Soll, Ist, and fulfillment percentage

In [7]:
def numeric_or_none(value):
    if value is None or value == "":
        return None
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    text = clean_text(value).replace("%", "").replace(",", ".")
    try:
        return float(text)
    except ValueError:
        return None


def extract_indicator_kpi(workbook_path, sheet_name="EB", indicator_query="Anzahl der Projektteilnehmenden gesamt"):
    wb = load_workbook(workbook_path, data_only=True, read_only=False)
    ws = wb[sheet_name]

    row_candidates = []
    for row in range(1, min(ws.max_row, 120) + 1):
        label = clean_text(ws.cell(row=row, column=3).value)
        if label:
            row_candidates.append((row, label))

    best_row = cosine_best(indicator_query, [label for _, label in row_candidates], min_score=0.55)
    if best_row is None:
        raise ValueError("Could not find indicator row")
    row_idx, row_score = best_row
    indicator_row, matched_indicator = row_candidates[row_idx]

    header_cells = []
    # Header information is spread over rows 9 and 10 in this template.
    for col in range(1, ws.max_column + 1):
        text = " ".join(clean_text(ws.cell(row=r, column=col).value) for r in [9, 10]).strip()
        text = re.sub(r"\s+", " ", text)
        if text:
            header_cells.append((col, text))

    def best_col(german_query, min_score=0.45):
        best = cosine_best(german_query, [text for _, text in header_cells], min_score=min_score)
        if best is None:
            return None, None, 0.0
        idx, score = best
        return header_cells[idx][0], header_cells[idx][1], score

    soll_col, soll_header, soll_score = best_col("Anzahl Soll laut Vertrag", min_score=0.45)
    ist_col, ist_header, ist_score = best_col("Anzahl Ist gesamt erreicht", min_score=0.45)
    pct_col, pct_header, pct_score = best_col("Erfüllt in Prozent Zielerreichung", min_score=0.45)

    soll = numeric_or_none(ws.cell(row=indicator_row, column=soll_col).value)
    ist = numeric_or_none(ws.cell(row=indicator_row, column=ist_col).value)
    fulfillment = numeric_or_none(ws.cell(row=indicator_row, column=pct_col).value)
    if fulfillment is not None and fulfillment <= 2:
        fulfillment_percent = fulfillment * 100
    else:
        fulfillment_percent = fulfillment

    return {
        "indikator": matched_indicator,
        "soll": int(soll) if soll is not None and soll.is_integer() else soll,
        "ist": int(ist) if ist is not None and ist.is_integer() else ist,
        "erfuellung_prozent": round(fulfillment_percent, 1) if fulfillment_percent is not None else None,
        "confidence": round(min(row_score, soll_score, ist_score, pct_score), 3),
        "evidence": {
            "source_file": str(workbook_path.relative_to(ROOT)),
            "sheet": sheet_name,
            "indicator_row": indicator_row,
            "matched_indicator": matched_indicator,
            "soll_header": soll_header,
            "ist_header": ist_header,
            "percent_header": pct_header,
        },
    }


main_kpi = extract_indicator_kpi(IST_XLSX)
main_kpi

/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


{'indikator': 'Anzahl der Projektteilnehmenden gesamt',
 'soll': 180,
 'ist': 194,
 'erfuellung_prozent': 107.8,
 'confidence': 0.755,
 'evidence': {'source_file': 'Hackathon/2024_2025/042_HandWerk.Zukunft-2024-2025_Indikatorenbericht.xlsx',
  'sheet': 'EB',
  'indicator_row': 11,
  'matched_indicator': 'Anzahl der Projektteilnehmenden gesamt',
  'soll_header': 'Anzahl SOLL lt. Vertrag',
  'ist_header': 'Anzahl IST - gesamt',
  'percent_header': 'Erfüllt in %'}}

## 5. Parse Word documents and extract German entities with GLiNER

In [8]:
def docx_text_blocks(path):
    doc = Document(path)
    blocks = []

    for idx, para in enumerate(doc.paragraphs):
        text = clean_text(para.text)
        if text:
            blocks.append({
                "source_file": str(path.relative_to(ROOT)),
                "kind": "paragraph",
                "index": idx,
                "style": para.style.name,
                "text": text,
            })

    for table_idx, table in enumerate(doc.tables):
        cell_texts = []
        for row in table.rows:
            for cell in row.cells:
                text = clean_text(cell.text)
                if text:
                    cell_texts.append(text)
        if cell_texts:
            blocks.append({
                "source_file": str(path.relative_to(ROOT)),
                "kind": "table",
                "index": table_idx,
                "style": None,
                "text": " | ".join(cell_texts),
            })

    return blocks


word_blocks = docx_text_blocks(PROJECT_DESCRIPTION_DOCX) + docx_text_blocks(FINAL_REPORT_DOCX)
word_df = pd.DataFrame(word_blocks)
word_df.head(10)

,source_file,kind,index,style,text
0,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,paragraph,7,No Spacing,Nationale Integrationsförderung 2024 und 2025
1,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,paragraph,11,Normal,PROJEKTBESCHREIBUNG
2,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,paragraph,13,Normal,&
3,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,paragraph,14,No Spacing,Inhaltsverzeichnis
4,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,paragraph,16,Heading 1,Kurzdarstellung des Projektes
5,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,paragraph,17,No Spacing,Geben Sie eine strukturierte und präzise Übers...
6,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,paragraph,19,No Spacing,Projekttitel
7,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,paragraph,21,No Spacing,Projektinhalt kurz zusammengefasst: (maximal 5...
8,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,paragraph,23,No Spacing,Besonderheiten der Zielgruppe: (maximal 2 Zeilen)
9,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,paragraph,25,No Spacing,Ziel(e) des Projektes: (maximal 2-3 Zeilen)


In [9]:
# German GLiNER labels. Keep these labels domain-specific and German.
gliner_labels_de = [
    "Zielgruppe",
    "Projektmaßnahme",
    "Projektziel",
    "Wirkung",
    "Region",
    "Kooperationspartner",
]

# Use only relevant text blocks to keep inference fast.
relevant_terms = [
    "zielgruppe", "teilnehm", "maßnahme", "projektinhalt", "ziel", "wirkung",
    "beratung", "kurs", "praktika", "lehr", "region", "standort"
]
relevant_blocks = [
    block for block in word_blocks
    if any(term in block["text"].lower() for term in relevant_terms)
]

entities = []
for block in relevant_blocks:
    text = block["text"][:3500]
    predictions = gliner_model.predict_entities(text, gliner_labels_de, threshold=0.35)
    for pred in predictions:
        entities.append({
            "label": pred["label"],
            "text": clean_text(pred["text"]),
            "score": round(float(pred["score"]), 3),
            "source_file": block["source_file"],
            "block_kind": block["kind"],
            "block_index": block["index"],
        })

entities_df = pd.DataFrame(entities).drop_duplicates() if entities else pd.DataFrame()
entities_df.sort_values(["label", "score"], ascending=[True, False]).head(30)

/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 550 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 427 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 537 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/peter/Dokumente/hackathon/PublicAIHackathonTeam3/.venv/lib64/python3.14/site-packages/gliner/data_processing/pro

,label,text,score,source_file,block_kind,block_index
252,Kooperationspartner,AMS Steiermark,0.820,Hackathon/2024_2025/042_HandWerk.Zukunft-2024_...,table,6
189,Kooperationspartner,zahlreichen kooperierenden Unternehmen,0.794,Hackathon/2024_2025/042_HandWerk.Zukunft-2024_...,paragraph,48
148,Kooperationspartner,Rote Kreuz Steiermark,0.747,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,23
109,Kooperationspartner,AMS,0.743,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,15
104,Kooperationspartner,AMS,0.716,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,15
210,Kooperationspartner,Projektpartnerschaften,0.709,Hackathon/2024_2025/042_HandWerk.Zukunft-2024_...,paragraph,67
105,Kooperationspartner,Alpha Nova Jugendcoaching,0.708,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,15
111,Kooperationspartner,bit,0.675,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,15
106,Kooperationspartner,bit,0.656,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,15
223,Kooperationspartner,AMS,0.640,Hackathon/2024_2025/042_HandWerk.Zukunft-2024_...,table,1


## 6. Use embeddings to select the best German Word evidence for target group and measures

GLiNER gives candidate spans. Embeddings choose the most relevant source blocks for our final prototype fields.

In [10]:
def best_text_blocks(german_query, blocks, top_k=3, min_score=0.35):
    texts = [block["text"] for block in blocks]
    query_embedding = embedding_model.encode(german_query, convert_to_tensor=True, normalize_embeddings=True)
    block_embeddings = embedding_model.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(query_embedding, block_embeddings)[0]
    ranked = sorted(enumerate(scores), key=lambda item: float(item[1]), reverse=True)
    results = []
    for idx, score in ranked[:top_k]:
        score = float(score)
        if score >= min_score:
            block = blocks[idx]
            results.append({
                "score": round(score, 3),
                "source_file": block["source_file"],
                "block_kind": block["kind"],
                "block_index": block["index"],
                "text": block["text"],
            })
    return results


target_group_evidence = best_text_blocks(
    "Zielgruppe des Projekts: Jugendliche und junge Erwachsene mit Migrationshintergrund, die eine Lehre beginnen wollen",
    word_blocks,
    top_k=3,
)

measures_evidence = best_text_blocks(
    "umzusetzende Projektmaßnahmen: Beratung, Vorqualifizierung, Deutschkurs, Workshops, Praktika, Lehrstellenvermittlung",
    word_blocks,
    top_k=3,
)

pd.DataFrame(target_group_evidence + measures_evidence)[["score", "source_file", "block_kind", "block_index", "text"]]

,score,source_file,block_kind,block_index,text
0,0.699,Hackathon/2024_2025/042_HandWerk.Zukunft-2024_...,table,0,1. Zwischenbericht Die vertraglich vereinbarte...
1,0.647,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,3,Das Ziel der 3-monatigen Qualifizierungsmaßnah...
2,0.647,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,24,Ziel 1: Verbesserung der fachspezifischen Deut...
3,0.736,Hackathon/2024_2025/042_HandWerk.Zukunft-2024_...,paragraph,43,Im Projektzeitraum wurden in „HandWerk.Zukunft...
4,0.721,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,table,4,Im Projektzeitraum sollen 6 Maßnahmen umgesetz...
5,0.695,Hackathon/2024_2025/042_HandWerk.Zukunft_2024_...,paragraph,107,3.1.3. Erfahrung in der Projektabwicklung


## 7. Build the final prototype record

For the prototype we keep a concise, SQL-friendly record and preserve evidence separately.

In [11]:
def top_entities(label, n=5):
    if entities_df.empty:
        return []
    rows = entities_df[entities_df["label"] == label].sort_values("score", ascending=False).head(n)
    return rows["text"].tolist()


prototype_record = {
    "projektnummer": project_meta["projektnummer"]["value"],
    "projekttitel": project_meta["projekttitel"]["value"],
    "projekttraeger": project_meta["projekttraeger"]["value"],
    "laufzeit_beginn": project_meta["laufzeit_beginn"]["value"],
    "laufzeit_ende": project_meta["laufzeit_ende"]["value"],
    "hauptprojektstandort": project_meta["hauptprojektstandort"]["value"],
    "zielgruppe_gliner_kandidaten": top_entities("Zielgruppe", n=5),
    "projektmassnahmen_gliner_kandidaten": top_entities("Projektmaßnahme", n=8),
    "hauptindikator_name": main_kpi["indikator"],
    "hauptindikator_soll": main_kpi["soll"],
    "hauptindikator_ist": main_kpi["ist"],
    "hauptindikator_erfuellung_prozent": main_kpi["erfuellung_prozent"],
    "zielgruppe_beste_textstelle": target_group_evidence[0]["text"] if target_group_evidence else None,
    "massnahmen_beste_textstelle": measures_evidence[0]["text"] if measures_evidence else None,
}

print(json.dumps(prototype_record, ensure_ascii=False, indent=2))

{
  "projektnummer": "042-2024/25",
  "projekttitel": "HandWerk.Zukunft – Berufsvorbereitungskurse mit Lehrvermittlung",
  "projekttraeger": "Steiermark",
  "laufzeit_beginn": "2024-01-01",
  "laufzeit_ende": "2025-12-31",
  "hauptprojektstandort": "Steiermark",
  "zielgruppe_gliner_kandidaten": [
    "Menschen mit Migrationsbiographie",
    "Menschen mit Migrationsbiographie",
    "Zielgruppe",
    "Menschen mit Migrationshintergrund",
    "Menschen"
  ],
  "projektmassnahmen_gliner_kandidaten": [
    "Maßnahme 9",
    "Maßnahme 7",
    "Maßnahme 6",
    "Evaluierungen",
    "Maßnahme 8",
    "Supervision",
    "Projekt",
    "Maßnahme 2"
  ],
  "hauptindikator_name": "Anzahl der Projektteilnehmenden gesamt",
  "hauptindikator_soll": 180,
  "hauptindikator_ist": 194,
  "hauptindikator_erfuellung_prozent": 107.8,
  "zielgruppe_beste_textstelle": "1. Zwischenbericht Die vertraglich vereinbarten Zielgruppen wurden erfolgreich erreicht. Im Fokus standen Jugendliche und junge Erwachsene im

In [12]:
evidence = {
    "project_meta": project_meta,
    "main_kpi": main_kpi,
    "target_group_evidence": target_group_evidence,
    "measures_evidence": measures_evidence,
    "gliner_entities": entities,
}

output_dir = ROOT / "notebooks" / "outputs"
output_dir.mkdir(exist_ok=True)

record_path = output_dir / "042_2024_25_prototype_record.json"
evidence_path = output_dir / "042_2024_25_extraction_evidence.json"

record_path.write_text(json.dumps(prototype_record, ensure_ascii=False, indent=2), encoding="utf-8")
evidence_path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print("Wrote:")
print(record_path.relative_to(ROOT))
print(evidence_path.relative_to(ROOT))

Wrote:
notebooks/outputs/042_2024_25_prototype_record.json
notebooks/outputs/042_2024_25_extraction_evidence.json


## 8. SQL-shaped output

This is the shape that can later be inserted into a SQL table such as `project_extraction_prototype`.

In [13]:
sql_ready_df = pd.DataFrame([{
    "project_number": prototype_record["projektnummer"],
    "project_name": prototype_record["projekttitel"],
    "organization": prototype_record["projekttraeger"],
    "runtime_start": prototype_record["laufzeit_beginn"],
    "runtime_end": prototype_record["laufzeit_ende"],
    "region": prototype_record["hauptprojektstandort"],
    "main_indicator_name": prototype_record["hauptindikator_name"],
    "main_indicator_soll": prototype_record["hauptindikator_soll"],
    "main_indicator_ist": prototype_record["hauptindikator_ist"],
    "main_indicator_fulfillment_percent": prototype_record["hauptindikator_erfuellung_prozent"],
    "target_group_text": prototype_record["zielgruppe_beste_textstelle"],
    "measures_text": prototype_record["massnahmen_beste_textstelle"],
}])

sql_ready_df

,project_number,project_name,organization,runtime_start,runtime_end,region,main_indicator_name,main_indicator_soll,main_indicator_ist,main_indicator_fulfillment_percent,target_group_text,measures_text
0,042-2024/25,HandWerk.Zukunft – Berufsvorbereitungskurse mi...,Steiermark,2024-01-01,2025-12-31,Steiermark,Anzahl der Projektteilnehmenden gesamt,180,194,107.8,1. Zwischenbericht Die vertraglich vereinbarte...,Im Projektzeitraum wurden in „HandWerk.Zukunft...


In [14]:
from IPython.display import Markdown, display


def format_sql_ready_row(df):
    row = df.iloc[0].to_dict()
    lines = ["# SQL-ready extraction result", ""]
    for field, value in row.items():
        if pd.isna(value):
            value = ""
        value = str(value).strip()
        lines.append(f"## `{field}`")
        lines.append(value if value else "_empty_")
        lines.append("")
    return "\n".join(lines)


display(Markdown(format_sql_ready_row(sql_ready_df)))


# SQL-ready extraction result

## `project_number`
042-2024/25

## `project_name`
HandWerk.Zukunft – Berufsvorbereitungskurse mit Lehrvermittlung

## `organization`
Steiermark

## `runtime_start`
2024-01-01

## `runtime_end`
2025-12-31

## `region`
Steiermark

## `main_indicator_name`
Anzahl der Projektteilnehmenden gesamt

## `main_indicator_soll`
180

## `main_indicator_ist`
194

## `main_indicator_fulfillment_percent`
107.8

## `target_group_text`
1. Zwischenbericht Die vertraglich vereinbarten Zielgruppen wurden erfolgreich erreicht. Im Fokus standen Jugendliche und junge Erwachsene im Alter von 15 bis 25 Jahren mit Migrationsbiographie. Die Zielgruppe setzt sich wie folgt zusammen (inkl. laufender Kurs): Drittstaatsangehörige mit längerfristiger Aufenthaltsperspektive: 13 Personen Asylberechtigte und subsidiär Schutzberechtigte: 9 Personen EU-Bürgerinnen und EU-Bürger: 4 Personen Österreicherinnen und Österreicher mit Migrationshintergrund: 4 Personen Altersstruktur der Teilnehmenden Der Altersdurchschnitt lag sowohl im ersten als auch im zweiten Durchgang bei 19 Jahren. In dem noch laufenden Durchgang sind jedoch mehr 17-jährige Teilnehmende vertreten. Die an den häufigsten vertretenen Altersgruppen sind die 17- sowie die 18- bis 19-Jährigen. Geschlechterverteilung Von den insgesamt 30 Teilnehmenden waren: Männer: 20 (66%) Frauen: 10 (33%) Herkunftsländer Die häufigsten Herkunftsländer der Teilnehmenden sind: Syrien Türkei Bosnien und Herzegowina Afghanistan Österreich 2. Zwischenbericht In der weiteren Projektperiode, das bedeutet auch in der dritten Kursmaßnahme konnte, die im Fördervertrag definierte Zielgruppe, bestehend aus Jugendlichen und jungen Erwachsenen im Alter von 15 bis 25 Jahren mit Migrationsbiografie erfolgreich erreicht werden. Die Zahl der Teilnehmenden erhöhte sich auf insgesamt 47, wobei sich die Zusammensetzung wie folgt verändert hat: Drittstaatsangehörige mit längerfristiger Aufenthaltsperspektive: 15 Personen Asylberechtigte und subsidiär Schutzberechtigte: 19 Personen EU-Bürgerinnen und EU-Bürger: 5 Personen Österreicherinnen und Österreicher mit Migrationshintergrund: 8 Personen Es zeigt sich, dass die Anzahl an Asylberechtigten mit subsidiär Schutzberechtigten sowie Drittstaatsangehörigen mit längerfristiger Aufenthaltsperspektive leicht gestiegen ist. Auch die Anzahl an Österreicher:innen mit Migrationshintergrund und EU-Bürger:innen war stärker vertreten. Ein auffälliger Unterschied im Vergleich zu den ersten beiden Kursmaßnahmen zeigte sich in der Altersstruktur. Der Altersdurchschnitt liegt nach wie vor bei rund 19 Jahren, wobei im dritten Kursdurchgang deutlich mehr 16-jährige Jugendliche vertreten waren. Diese jüngere Altersgruppe brachte auch einen höheren Orientierungsbedarf mit, was sich darin zeigte, dass mehr Praktika absolviert wurden, um herauszufinden was einem liegt und was nicht. Die Geschlechterverteilung veränderte sich ebenfalls, so stieg der Anteil der männlichen Teilnehmenden im Vergleich zu den ersten beiden Kursmaßnahmen auf 74,5%, während der Anteil weiblicher Teilnehmender auf 25,5% sank. Die häufigsten Herkunftsländer der Teilnehmenden blieben im Wesentlichen gleich. Angeführt wird die Liste nach wie vor von Syrien (12), Österreich (7), Afghanistan (6), der Türkei (4) und Bosnien und Herzegowina (3), andere (15). 3. Zwischenbericht Die vierte Kursmaßnahme wurde bereits abgeschlossen, die fünfte läuft noch bis 08.08.2025. Im dritten Zwischenbericht werden beide Kurse gemeinsam dargestellt. Die Zielgruppe umfasste 11 Drittstaatsangehörige mit längerfristiger Aufenthaltsperspektive, 6 Asyl- bzw. subsidiär Schutzberechtigte, 10 EU-Bürger:innen und 5 Österreicher:innen mit Migrationshintergrund. Der Anteil an Drittstaatsangehörigen und EU-Bürger:innen war höher als in den vorangegangenen Kursen. Der Altersdurchschnitt lag im vierten Kurs bei 18 Jahren, im fünften bei 17 Jahren. Es nahmen erneut überwiegend sehr junge Personen zwischen 16 und 18 Jahren teil. Die Geschlechterverteilung blieb konstant: Im Kurs 4 nahmen 11 Männer und 5 Frauen teil, im Kurs 5 bisher 12 Männer und 4 Frauen. Die häufigsten Herkunftsländer waren die Türkei, Tschetschenien und Afghanistan. 4. Endbericht Die sechste Kursmaßnahme umfasste insgesamt 5 Drittstaatsangehörige mit längerfristiger Aufenthaltsperspektive, 6 Asyl- bzw. subsidiär Schutzberechtigte, 7 EU-Bürger:innen und 1 Österreicherin mit Migrationshintergrund. Der Altersdurchschnitt im 6. Kurs lag bei 17 Jahren. Die Geschlechterverteilung blieb konstant: Im 6. Kurs nahmen 14 Männer und 5 Frauen teil. Die häufigsten Herkunftsländer waren Rumänien, Syrien und Ägypten, jeweils mit 3 Teilnehmer:innen. Insgesamt lässt sich festhalten, dass im gesamten Projektverlauf 98 Jugendliche und junge Erwachsene erreicht wurden. Die Teilnehmenden kamen überwiegend aus Drittstaaten bzw. verfügten über einen Asyl- oder subsidiären Schutzstatus. Darüber hinaus nahmen auch EU-Bürger:innen sowie Österreicher:innen mit Migrationshintergrund an den Kursmaßnahmen teil. Die häufigsten Herkunftsländer waren Syrien und Afghanistan. Der Altersdurchschnitt lag über alle Kursmaßnahmen hinweg bei rund 17 bis 19 Jahren, wobei in den späteren Kursdurchgängen vermehrt jüngere Teilnehmende zwischen 16 und 18 Jahren vertreten waren. Die Geschlechterverteilung war durchgehend männlich dominiert, mit einem Anteil von rund 70 % Männern und etwa 30 % Frauen.

## `measures_text`
Im Projektzeitraum wurden in „HandWerk.Zukunft – Berufsvorbereitungskurse mit Lehrvermittlung 6 Kursmaßnahmen (Modul 2) planmäßig umgesetzt. In den jeweils 12 Wochen dauernden Kursmaßnahmen fanden wöchentlich wechselnd fachsprachlicher Unterricht und Workshops statt, begleitet von Mentoring, kursbegleitender Beratung und Unterstützung sowie Reflexions- und Follow-up-Gesprächen, psychosoziale Unterstützung und Nachbetreuung (Module 3 und 4). Der fachsprachliche Unterricht orientierte sich an den Kenntnissen der Teilnehmenden und den Berufswünschen. Die Workshops gliederten sich in die Themenbereiche Berufsbilder, Zeitmanagement, Telefon- und Kommunikationstraining, Bewerbungstraining, berufsspezifische Grundfertigkeiten, Mathematik und Office-Anwendertraining.
